# Construction of the Synthetic Evaluation Dataset

**Purpose:** Aggregate and select 50 questions from document-specific synthetic test sets for evaluation.

**Objective:** In this notebook, we consolidate the individual synthetic test sets generated from our document corpus. We filter for specific query characteristics (Short Queries), create a randomized sample, and perform a manual quality assurance step by swapping out unsuitable questions with a reserve "backup" pool.

In [ ]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/processed/testset")
exporting_path = Path("../data/evaluation")

## 1. Data Loading

We iterate through all CSV files in the processed test set directory. A `source_file` column is added to track the origin of each question. Finally, all dataframes are concatenated into a single master dataframe.

In [ ]:
### Load Document-Specific Synthetic Testsets

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# Steps 1, 2 & 3 combined in a compact list comprehension
# We use .assign() to create the 'source_file' column directly during loading
dfs = [
    pd.read_csv(f).assign(source_file=f.name) 
    for f in data_path.glob("*.csv")
]

# Step 4: Combine all dataframes
df = pd.concat(dfs, ignore_index=True)

# Validation check
print(f"Shape: {df.shape}")

## 2. Initial Data Exploration

We briefly analyze the distribution of user input lengths to understand the dataset's characteristics before filtering.

In [ ]:
df_shorted = df.assign(user_input_length=df['user_input'].str.len()).sort_values('user_input_length', ascending=False)

df_shorted = df_shorted.head(50)

num_short = (df['query_length'] == 'SHORT').sum()
print("Anzahl Short Questions:", num_short)

## 3. Sampling and Backup Creation

We shuffle the dataset to ensure a random distribution of topics. We then separate the data into:
1.  **Final 50:** The primary candidate set for evaluation.
2.  **Backup Questions:** A reserve pool used to replace any low-quality questions found in the primary set.

In [ ]:
short_df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)
final_50 = short_df_shuffled.head(50)
backup_questions = short_df_shuffled.iloc[50:]
final_50['source_file'].nunique()
final_50[['user_input','reference']]

doc_counts = final_50['source_file'].value_counts()
print(doc_counts)

## 4. Refining Selection (Short Queries Only)

To ensure a focused evaluation, we strictly filter the dataset to include only questions categorized as `SHORT`. We then re-sample the final 50 candidates and the backup pool from this filtered subset.

In [ ]:
short_df = df[(df['query_length'] == 'SHORT')]
short_df_shuffled = short_df.sample(frac=1, random_state=42).reset_index(drop=True)
final_50 = short_df_shuffled.head(50).copy()
backup_questions = short_df_shuffled.iloc[50:].copy()

final_50[['user_input','reference']]

## 5. Manual Curation and Swapping

In this step, we review the selected questions. If a question is deemed unsuitable (e.g., ambiguous, duplicate, or low quality), we manually replace it with the next available question from the `backup_questions` pool.

**Swap 1:** Replacing Index 1.

In [ ]:
# 2. Display the swap candidates (sanity check)
print("Old (Index 1):", final_50.iloc[1]['user_input'])
print("New (Backup 0):", backup_questions.iloc[0]['user_input'])

# 3. Perform the swap: Overwrite row 1 with the first row from backup
final_50.iloc[1] = backup_questions.iloc[0]

# 4. Verification: Is the new value in place?
print("Current Index 1:", final_50.iloc[1]['user_input'])

**Swap 2:** Replacing Index 4.

In [ ]:
# 2. Display the swap candidates (sanity check)
print("Old (Index 4):", final_50.iloc[4]['user_input'])
print("New (Backup 1):", backup_questions.iloc[1]['user_input'])

# 3. Perform the swap: Overwrite row 4 with the second row from backup
final_50.iloc[4] = backup_questions.iloc[1]

# 4. Verification: Is the new value in place?
print("Current Index 4:", final_50.iloc[4]['user_input'])

**Swap 3:** Replacing Index 10.

In [ ]:
# 2. Display the swap candidates (sanity check)
print("Old (Index 10):", final_50.iloc[10]['user_input'])
print("New (Backup 2):", backup_questions.iloc[2]['user_input'])

# 3. Perform the swap: Overwrite row 10 with the third row from backup
final_50.iloc[10] = backup_questions.iloc[2]

# 4. Verification: Is the new value in place?
print("Current Index 10:", final_50.iloc[10]['user_input'])

**Swap 4:** Replacing Index 44.

In [ ]:
# 2. Display the swap candidates (sanity check)
print("Old (Index 44):", final_50.iloc[44]['user_input'])
print("New (Backup 3):", backup_questions.iloc[3]['user_input'])

# 3. Perform the swap: Overwrite row 44 with the fourth row from backup
final_50.iloc[44] = backup_questions.iloc[3]

# 4. Verification: Is the new value in place?
print("Current Index 44:", final_50.iloc[44]['user_input'])

**Swap 5:** Replacing Index 47.

In [ ]:
# 2. Display the swap candidates (sanity check)
print("Old (Index 47):", final_50.iloc[47]['user_input'])
print("New (Backup 4):", backup_questions.iloc[4]['user_input'])

# 3. Perform the swap: Overwrite row 47 with the fifth row from backup
final_50.iloc[47] = backup_questions.iloc[4]

# 4. Verification: Is the new value in place?
print("Current Index 47:", final_50.iloc[47]['user_input'])

final_50[['user_input','reference']]

**Swap 6:** A second iteration on Index 1 (Replacing the previous replacement).

In [ ]:
# 2. Display the swap candidates (sanity check)
print("Old (Index 1):", final_50.iloc[1]['user_input'])
print("New (Backup 5):", backup_questions.iloc[5]['user_input'])

# 3. Perform the swap: Overwrite row 1 with the sixth row from backup
final_50.iloc[1] = backup_questions.iloc[5]

# 4. Verification: Is the new value in place?
print("Current Index 1:", final_50.iloc[1]['user_input'])

final_50[['user_input','reference']]

#final_50.to_csv("../data/evaluation/evaluation_testset_50_short_query_length.csv", index=False)